#### Building off of the 3D Semantic Segmentation notebook to perform Sensor Fusion between LiDAR and Camera, with the Waymo Open Dataset

##### By Jacob Igo

In [1]:
#library imports
import gcsfs
from google.cloud import storage
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import pandas as pd
import tensorflow as tf
import open3d as o3d
import numpy as np
import gc

#function imports
import semseg_functions

/home/jacob/projects/waymo_research/3d_semseg/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jacob/projects/waymo_research/3d_semseg/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:1781

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
#get google cloud token
import os
os.environ["CLOUDSDK_CONFIG"] = "/home/jacob/.config/gcloud"

import subprocess
token = subprocess.check_output(
    ["/usr/bin/gcloud", "auth", "print-access-token"]
).decode().strip()

In [3]:
from datetime import datetime, timezone, timedelta
fs = pafs.GcsFileSystem(access_token=token, credential_token_expiration=datetime.now(timezone.utc) + timedelta(hours=1))


camera_img_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/camera_image/"))
camera_calib_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/camera_calibration/"))

camera_img_file = pq.ParquetFile(camera_img_files[0].path, filesystem=fs)
camera_img_rg_df = camera_img_file.read_row_group(0).to_pandas()
print(camera_img_files[0].path)
camera_img_rg_df.head(5)

waymo_open_dataset_v_2_0_0/training/camera_image/10017090168044687777_6380_000_6400_000.parquet


,key.segment_context_name,key.frame_timestamp_micros,key.camera_name,[CameraImageComponent].image,[CameraImageComponent].pose.transform,[CameraImageComponent].velocity.linear_velocity.x,[CameraImageComponent].velocity.linear_velocity.y,[CameraImageComponent].velocity.linear_velocity.z,[CameraImageComponent].velocity.angular_velocity.x,[CameraImageComponent].velocity.angular_velocity.y,[CameraImageComponent].velocity.angular_velocity.z,[CameraImageComponent].pose_timestamp,[CameraImageComponent].rolling_shutter_params.shutter,[CameraImageComponent].rolling_shutter_params.camera_trigger_time,[CameraImageComponent].rolling_shutter_params.camera_readout_done_time
index,,,,,,,,,,,,,,,
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,1,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,"[0.9482868309813268, -0.23495154098744556, 0.2...",5.765131,1.371270,-1.329111,-0.010016,-0.014922,0.105475,1.550083e+09,0.009992,1.550083e+09,1.550083e+09
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,2,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,"[0.9487161220058431, -0.23293730459068215, 0.2...",5.757565,1.392409,-1.327417,0.003401,-0.011684,0.099375,1.550083e+09,0.009992,1.550083e+09,1.550083e+09
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,4,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,"[0.9489391880311817, -0.23197163266142976, 0.2...",5.762588,1.389895,-1.330847,0.005329,-0.012516,0.098726,1.550083e+09,0.009992,1.550083e+09,1.550083e+09
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,3,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,"[0.9480805733268344, -0.23597955012227076, 0.2...",5.764107,1.374694,-1.330062,-0.014756,-0.016971,0.102667,1.550083e+09,0.009992,1.550083e+09,1.550083e+09
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,5,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,"[0.9479157488563362, -0.23691579285250022, 0.2...",5.758528,1.381121,-1.324764,-0.016695,-0.015307,0.097743,1.550083e+09,0.009992,1.550083e+09,1.550083e+09


In [4]:
del camera_img_file, camera_img_rg_df
gc.collect()

20

In [5]:
camera_calib_file = pq.ParquetFile(camera_calib_files[0].path, filesystem=fs)
camera_calib_rg_df = camera_calib_file.read_row_group(0).to_pandas()
print(camera_calib_files[0].path)
camera_calib_rg_df.head(10)

waymo_open_dataset_v_2_0_0/training/camera_calibration/10017090168044687777_6380_000_6400_000.parquet


,key.segment_context_name,key.camera_name,[CameraCalibrationComponent].intrinsic.f_u,[CameraCalibrationComponent].intrinsic.f_v,[CameraCalibrationComponent].intrinsic.c_u,[CameraCalibrationComponent].intrinsic.c_v,[CameraCalibrationComponent].intrinsic.k1,[CameraCalibrationComponent].intrinsic.k2,[CameraCalibrationComponent].intrinsic.p1,[CameraCalibrationComponent].intrinsic.p2,[CameraCalibrationComponent].intrinsic.k3,[CameraCalibrationComponent].extrinsic.transform,[CameraCalibrationComponent].width,[CameraCalibrationComponent].height,[CameraCalibrationComponent].rolling_shutter_direction
0,10017090168044687777_6380_000_6400_000,1,2059.612012,2059.612012,952.412190,634.587208,0.035453,-0.338301,0.000019,0.000714,0.0,"[0.9999785086634438, 0.003142074430868787, 0.0...",1920,1280,2
1,10017090168044687777_6380_000_6400_000,2,2046.634611,2046.634611,975.056044,640.908707,0.030496,-0.310172,0.002555,0.000886,0.0,"[0.7182572077098084, -0.695773694450251, -0.00...",1920,1280,2
2,10017090168044687777_6380_000_6400_000,3,2053.547383,2053.547383,944.359788,630.649098,0.033324,-0.301891,0.000009,-0.000189,0.0,"[0.710693728813626, 0.703485222524296, 0.00479...",1920,1280,2
3,10017090168044687777_6380_000_6400_000,4,2050.252538,2050.252538,970.308559,248.135537,0.033466,-0.336335,0.000224,0.002455,0.0,"[0.010163829827457116, -0.9999065078473018, -0...",1920,886,2
4,10017090168044687777_6380_000_6400_000,5,2053.615601,2053.615601,970.546937,235.615792,0.026367,-0.288014,-0.000022,-0.000825,0.0,"[-0.0022351670479722214, 0.9999727460755228, 0...",1920,886,2


In [6]:
#input in a certain timestamp's set of 5 camera values,
#along with the universal camera calibration data, 
#and the global LiDAR coordinates (for that timestamp)


#projection function

import io
from PIL import Image

def projection_onto_image(lidar_df, camera_df, lidar_calib_df, camera_calib_df, lidar_timestamp, camera_range=1):
    g_coords, masked_labels = semseg_functions.laser_process(laser_num=1, df=lidar_df, df_rgc=lidar_calib_df, df_seg=None, timestamp=lidar_timestamp)
    
    camera_tuples = []

    i = camera_range      #process 5 cameras per timestamp


    camera_extrensic_matrix = np.reshape(camera_calib_df["[CameraCalibrationComponent].extrinsic.transform"].loc[camera_calib_df["key.camera_name"] == i].iloc[0], (4, 4))

    #retreiving bytes of camera image for this timestamp
    camera_row_mask = (camera_df["key.camera_name"] == i) & (camera_df["key.frame_timestamp_micros"] == lidar_timestamp)
    camera_image = camera_df["[CameraImageComponent].image"].loc[camera_row_mask].iloc[0]
    image_bytes = io.BytesIO(camera_image)

    with Image.open(image_bytes) as img:
        camera_array = np.asarray(img.convert("RGB"))
    height, width = camera_array.shape[:2]

    g_coords_homo = np.column_stack((g_coords, np.ones(len(g_coords))))
    inv_camera_extrensic = np.linalg.inv(camera_extrensic_matrix)

    #putting 3D global points into 3D camera space
    g_coords_homo_T = np.transpose(g_coords_homo)
    camera_local_points_h_T = np.matmul(inv_camera_extrensic, g_coords_homo_T)
    camera_local_points_h = np.transpose(camera_local_points_h_T)

    camera_local_points = camera_local_points_h[:, :3]

    ## === REMEMBER ===
    ## In this dataset, the X axis is the plane pointing out of the camera; Z is vertical, Y is horizontal

    camera_mask_channel = camera_local_points[:, 0]
    camera_mask = camera_mask_channel > 0

    #apply the mask to only get X values greater than 0 (detected in front of the camera, not behind)
    camera_local_points_masked = camera_local_points[camera_mask]
    #camera_local_labels_masked = masked_labels[camera_mask]


    #retrive horizontal (u) and vertical (v) values for focal length and principal (center) point
    horiz_focal_length = camera_calib_df["[CameraCalibrationComponent].intrinsic.f_u"].loc[camera_calib_df["key.camera_name"] == i].iloc[0]
    vert_focal_length = camera_calib_df["[CameraCalibrationComponent].intrinsic.f_v"].loc[camera_calib_df["key.camera_name"] == i].iloc[0]
    horiz_center_point = camera_calib_df["[CameraCalibrationComponent].intrinsic.c_u"].loc[camera_calib_df["key.camera_name"] == i].iloc[0]
    vert_center_point = camera_calib_df["[CameraCalibrationComponent].intrinsic.c_v"].loc[camera_calib_df["key.camera_name"] == i].iloc[0]

    #slice to do perspective divide (divide by X, the depth in the image)
    Depth_X = camera_local_points_masked[:, 0]
    Left_Y = camera_local_points_masked[:, 1]
    Up_Z = camera_local_points_masked[:, 2]

    #pixel columns increase to the right and rows increase down, so we have to negate the left and up ratios
    Left_Y_div = (Left_Y / Depth_X) * -1
    Up_Z_div = (Up_Z / Depth_X) * -1

    #intrinsic transformations - scaling for focal and principal points (rounding (and casting to INT) for future indexing)
    Left_Y_col = np.round((Left_Y_div * horiz_focal_length) + horiz_center_point).astype(int)  #pixel column
    Up_Z_row = np.round((Up_Z_div * vert_focal_length) + vert_center_point).astype(int)      #pixel row

    
    #Out of Bounds masking: keeping points that only fall within the cameras FOV
    inbound_pixel_mask = (Left_Y_col >= 0) & (Left_Y_col < width) & (Up_Z_row >= 0) & (Up_Z_row < height)
    Left_Y_col_masked = Left_Y_col[inbound_pixel_mask]
    Up_Z_row_masked = Up_Z_row[inbound_pixel_mask]
    
    #apply the mask to the 3D camera points and labels, too
    #camera_pixel_labels = camera_local_labels_masked[inbound_pixel_mask]
    camera_pixel_3D_points = camera_local_points_masked[inbound_pixel_mask]

    rgb_values = camera_array[Up_Z_row_masked, Left_Y_col_masked]
    
    #returns for each camera, as a tuple in a list:
    #1. camera number
    #2. RGB color values for each point
    #3. pixel column array
    #4. pixel row array
    #5. 3D camera points
    #6. camera labels - EDIT: not using seg labels since they are too sparse when comparing with images - only using depth for classifying
    #7. raw JPEG bytes of the camera image (kept compressed to stay memory-light; decode on demand for overlays)
    camera_tuples.append((i, rgb_values, Left_Y_col_masked, Up_Z_row_masked, camera_pixel_3D_points, camera_image))
    
    return camera_tuples



In [ ]:

#aligning the lidar, lidar_segmentation, and camera_image files and timestamps
#returns a list of tuples with 

def lidar_camera_processor(lidar_file, camera_file, seg_file):


    # TIME_GAP_THRESH = 50000
    lidar_seg_camera = []
    seg_timestamp_set = set()
    
    lidar_calib_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar_calibration/{os.path.basename(lidar_file)}", filesystem=fs)
    lidar_calib_df = lidar_calib_pq.read_row_group(0).to_pandas()

    camera_calib_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/camera_calibration/{os.path.basename(camera_file)}", filesystem=fs)
    camera_calib_df = camera_calib_pq.read_row_group(0).to_pandas()

    lidar_pq = pq.ParquetFile(lidar_file, filesystem=fs)
    camera_pq = pq.ParquetFile(camera_file, filesystem=fs)

    seg_pq = pq.ParquetFile(seg_file, filesystem=fs)

    for k in range(seg_pq.num_row_groups):
        lidar_df = seg_pq.read_row_group(k, columns=["key.frame_timestamp_micros"])

    # for l_tup in lidar_timestamp_index:
    #     lidar_file, lidar_timestamp = l_tup
    #closest_stamp_gap = 100000000
        # for s_tup in seg_timestamp_index:

        #     seg_file, seg_timestamp = s_tup
        #     if seg_timestamp == lidar_timestamp and os.path.basename(lidar_file) == os.path.basename(seg_file):

        #lidar_df = semseg_functions.timestamp_aligner(lidar_file, lidar_timestamp)
        #     seg_df = semseg_functions.timestamp_aligner(seg_file, seg_timestamp)

    for j in range(lidar_pq.num_row_groups):
        lidar_df = lidar_pq.read_row_group(j).to_pandas()
        camera_df = camera_pq.read_row_group(j).to_pandas()
        camera_times = camera_pq.read_row_group(j, columns=["key.frame_timestamp_micros"]).to_pandas()
        seg_df = camera_pq.read_row_group(j).to_pandas()
        seg_times = camera_pq.read_row_group(j, columns=["key.frame_timestamp_micros"]).to_pandas()
        # closest_stamp = 100000000
        # closest_file = None

        for lidar_timestamp in lidar_df["key.frame_timestamp_micros"].loc[lidar_df["key.laser_name"] == 1].unique():

            # camera_timestamp = camera_df["key.frame_timestamp_micros"].loc[camera_df["key.camera_name"] == 1].iloc[0]

            # if lidar_timestamp == camera_timestamp:
            if camera_times["key.frame_timestamp_micros"].isin([lidar_timestamp]).any():

                camera_tuple_list = projection_onto_image(lidar_df, camera_df, lidar_calib_df, camera_calib_df, lidar_timestamp, camera_range=1)
                lidar_seg_camera.append(camera_tuple_list)



    return lidar_seg_camera

            # current_stamp_gap = np.abs(lidar_timestamp - camera_timestamp)
            # if current_stamp_gap <= TIME_GAP_THRESH and current_stamp_gap <= closest_stamp_gap and os.path.basename(lidar_file) == os.path.basename(camera_file):
            #     #camera_calib_file = os.path.join("waymo_open_dataset_v_2_0_0/training/camera_calibration/", os.path.basename(camera_file))
            #     #camera_calib_df_t = timestamp_aligner(camera_calib_file, camera_timestamp)
            #     # camera_df = semseg_functions.timestamp_aligner(camera_file, camera_timestamp)
            #     closest_stamp_gap = current_stamp_gap
            #     closest_stamp = camera_timestamp
            #     closest_file = camera_file
            #     #continue

            # if lidar_timestamp - camera_timestamp <= 0 and closest_stamp_gap <= TIME_GAP_THRESH:
            #     break

    #     if closest_file is not None:
    #         #lidar_seg_camera.append((lidar_df, seg_df, camera_df))
    #         #begin processing projection
    #         camera_df = semseg_functions.timestamp_aligner(closest_file, closest_stamp)
    #         camera_tuple_list = projection_onto_image(lidar_df, camera_df, lidar_calib_df, camera_calib_df, lidar_timestamp, camera_range=[1])
    #         lidar_seg_camera.append(camera_tuple_list)

    
    # return lidar_seg_camera
    


lidar_timestamp_index = semseg_functions.folder_file_indexer("training/lidar", start_folder_index=0, end_folder_index=1)
#seg_timestamp_index = semseg_functions.folder_file_indexer("training/lidar_segmentation", start_folder_index=0, end_folder_index=1)
camera_timestamp_index = semseg_functions.folder_file_indexer("training/camera_image", start_folder_index=0, end_folder_index=1)

lidar_file = lidar_timestamp_index[0][0]
camera_file = camera_timestamp_index[0][0]

lidar_camera = lidar_camera_processor(lidar_file, camera_file)
print(len(lidar_camera))


198


In [ ]:
#animation time!
#overlay the projected LiDAR points on the front camera image, colored by depth, one frame per matched timestamp

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import Video

CAMERA_NUM = 1     #front camera
MAX_DEPTH = 75     #meters; clip the colormap here so near/far contrast doesn't wash out
OUTPUT_PATH = "media/init_sensor_fusion.mp4"

#pull the front camera's tuple from each frame and decode its image
frames = []
for camera_tuple_list in lidar_camera:
    for tup in camera_tuple_list:
        cam_num, rgb_values, pixel_cols, pixel_rows, points_3d, jpeg_bytes = tup
        if cam_num == CAMERA_NUM:
            with Image.open(io.BytesIO(jpeg_bytes)) as img:
                image_array = np.asarray(img.convert("RGB"))
            depth = points_3d[:, 0]    #X is depth in camera space
            frames.append((image_array, pixel_cols, pixel_rows, depth))
            break

#set up the figure with the first frame
image0, cols0, rows0, depth0 = frames[0]
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(image0)
scat = ax.scatter(cols0, rows0, c=depth0, cmap="jet", vmin=0, vmax=MAX_DEPTH, s=2)
title = ax.set_title(f"Frame 0 / {len(frames) - 1} — LiDAR projected onto front camera, colored by depth")
ax.set_axis_off()
fig.colorbar(scat, ax=ax, label="Depth (m)", shrink=0.7)
plt.tight_layout()

def update(frame_idx):
    image_array, pixel_cols, pixel_rows, depth = frames[frame_idx]
    im.set_data(image_array)
    scat.set_offsets(np.column_stack((pixel_cols, pixel_rows)))
    scat.set_array(depth)
    title.set_text(f"Frame {frame_idx} / {len(frames) - 1} — LiDAR projected onto front camera, colored by depth")
    return im, scat, title

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=100,    # ~10 Hz, matching Waymo capture rate
    blit=False,      # image background changes every frame, so blitting won't help
)

#write straight to an mp4 with the ffmpeg writer (~10 fps) instead of embedding base64 frames
#in the notebook — this avoids matplotlib's animation.embed_limit dropping frames
writer = animation.FFMpegWriter(fps=10, bitrate=4000)
anim.save(OUTPUT_PATH, writer=writer, dpi=100)

plt.close(fig)    #suppress the static plot alongside the animation
del anim, frames
gc.collect()

#embed the saved file (loaded from disk, not base64'd into the notebook)
Video(OUTPUT_PATH, embed=False)

: 